# Build a private subnet that reaches S3 without a NAT Gateway and prove the cost delta

A real Reddit post from 2024 described a $9,700 monthly NAT Gateway bill driven almost entirely by S3 egress. The team did not need internet egress at all; they just routed S3 traffic through NAT because that is what the VPC wizard set up. You are the engineer doing the audit. The CFO is asking why a service that bills both per hour and per GB processed is sitting in the data path of every S3 PUT. You have one session to stand up both architectures, prove Architecture B does the same work without NAT, and produce the metric that backs the migration recommendation.

You will build two architectures side by side in the same VPC:

- **Architecture A.** Public subnet with a NAT Gateway plus an Elastic IP. A private subnet whose default route points at the NAT Gateway. An EC2 instance in that private subnet uploads 100 MB of payload to S3 through the NAT path.
- **Architecture B.** A second private subnet routed to a VPC Gateway Endpoint for S3. No 0.0.0.0/0 route anywhere on this side. An EC2 instance in this subnet uploads the same 100 MB of payload to S3, never touching NAT.

Then you read the CloudWatch `BytesOutToDestination` metric on the NAT Gateway and see that Architecture A pushed ~100 MB through NAT while Architecture B contributed zero NAT bytes.

**Time.** Plan for about 60 minutes hands-on plus a 5-minute CloudWatch settling window at the end. The session window is 120 minutes so the metric aggregation has room to publish.

**Cost.** This is the first SAA lab that costs more than a nickel. The NAT Gateway is the line item that matters: 4.5 cents per hour plus 4.5 cents per gigabyte processed. If you nail the cleanup at the end of the session you spend about 10 cents. If you walk away with the NAT Gateway running you spend about a dollar a day. The companion panel will start poking you if the kernel idles past 30 minutes with this critical resource alive.

This lab maps to AWS SAA-C03 Domain 1 (Design Secure Architectures, 30%) and Domain 4 (Design Cost-Optimized Architectures, 20%).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. Architecture suffixes -a and -b disambiguate the two
# competing architectures per blueprint section 21 (Compare-it).

import atexit
import base64
import datetime as dt
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "vpc-endpoints-vs-nat-egress"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"
AZ_PUBLIC = "us-east-1a"
AZ_PRIVATE_A = "us-east-1a"
AZ_PRIVATE_B = "us-east-1b"

VPC_CIDR = "10.42.0.0/16"
PUBLIC_SUBNET_CIDR = "10.42.0.0/24"
PRIVATE_SUBNET_A_CIDR = "10.42.1.0/24"
PRIVATE_SUBNET_B_CIDR = "10.42.2.0/24"

# Resource names (filled in after we know the account ID).
VPC_NAME = f"labrun-{LAB_ID}-vpc"
IGW_NAME = f"labrun-{LAB_ID}-igw"
PUBLIC_SUBNET_NAME = f"labrun-{LAB_ID}-public-subnet"
PRIVATE_SUBNET_A_NAME = f"labrun-{LAB_ID}-private-subnet-a"
PRIVATE_SUBNET_B_NAME = f"labrun-{LAB_ID}-private-subnet-b"
PUBLIC_RT_NAME = f"labrun-{LAB_ID}-public-rt"
PRIVATE_RT_A_NAME = f"labrun-{LAB_ID}-private-rt-a"
PRIVATE_RT_B_NAME = f"labrun-{LAB_ID}-private-rt-b"
SG_NAME = f"labrun-{LAB_ID}-sg"
EC2_A_NAME = f"labrun-{LAB_ID}-ec2-a"
EC2_B_NAME = f"labrun-{LAB_ID}-ec2-b"
EC2_ROLE_NAME = f"labrun-{LAB_ID}-ec2-role"
INSTANCE_PROFILE_NAME = f"labrun-{LAB_ID}-instance-profile"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-putobject-policy"

BUCKET_NAME = None  # resolved after STS get_caller_identity

# Runtime state captured as the lab proceeds. These are intentionally
# module-level so the checkpoints and observe cells can read them.
LAB_STATE = {
    "session_start": None,
    "vpc_id": None,
    "igw_id": None,
    "public_subnet_id": None,
    "private_subnet_a_id": None,
    "private_subnet_b_id": None,
    "public_rt_id": None,
    "private_rt_a_id": None,
    "private_rt_b_id": None,
    "sg_id": None,
    "eip_allocation_id": None,
    "nat_gateway_id": None,
    "vpc_endpoint_id": None,
    "instance_a_id": None,
    "instance_b_id": None,
}

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the $20 monthly Budget is in place. This lab is the SAA track's
# biggest cost risk so setup refuses to proceed if no Budget exists per
# RESOURCE-SAFETY-SPEC Lab 3 mitigation.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA Batch 2 labs run in {REGION} (N. Virginia).")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Budget prerequisite check. NAT Gateway is the biggest cost risk in SAA;
# setup refuses to provision if no monthly budget exists at $20 or below.
try:
    budgets = boto3.client(
        "budgets",
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token or None,
        region_name="us-east-1",
    )
    resp = budgets.describe_budgets(AccountId=ACCOUNT_ID, MaxResults=100)
    has_budget = False
    for b in resp.get("Budgets", []):
        limit = b.get("BudgetLimit", {})
        if limit.get("Unit", "").upper() == "USD" and float(limit.get("Amount", 0)) <= 20.0:
            has_budget = True
            break
    if not has_budget:
        print()
        print("BLOCKED: no AWS Budget at $20/month or below was found.")
        print("This lab provisions a NAT Gateway, which bills $0.045/hour.")
        print("Create a $20 monthly budget in the AWS console first:")
        print("  https://console.aws.amazon.com/billing/home#/budgets")
        print("Then re-run this cell.")
        raise SystemExit(1)
    print("Budget guard: at least one $20 monthly Budget found.")
except ClientError as e:
    code = e.response.get("Error", {}).get("Code", "")
    if code in ("AccessDeniedException", "NotAuthorizedException"):
        print("WARNING: cannot read AWS Budgets (AccessDenied). Skipping guard.")
        print("Confirm you have a $20 monthly budget configured before continuing.")
    else:
        raise

LAB_STATE["session_start"] = dt.datetime.now(dt.timezone.utc)
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest order is critical-first (per RESOURCE-SAFETY-SPEC Rule 2): EC2
# instances and NAT Gateway tear down before the network scaffold so the
# hourly meter stops as soon as possible. The NAT Gateway adapter waits for
# State=deleted before returning so the subsequent EIP release can succeed.

CLEANUP_MANIFEST = []


def _append_manifest(resource):
    CLEANUP_MANIFEST.append(resource)


def _rebuild_manifest():
    """Rebuild the manifest in critical-first reverse-creation order from
    the IDs captured in LAB_STATE. Called by every task cell after a resource
    is created so atexit always sees the latest snapshot."""
    CLEANUP_MANIFEST.clear()
    # S3 payload objects.
    if BUCKET_NAME:
        for prefix in ("from-instance-a", "from-instance-b"):
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="s3_object",
                    id="payload.bin",
                    parent=BUCKET_NAME,
                    region=REGION,
                    cli_delete_command=(
                        f"aws s3api delete-object --bucket {BUCKET_NAME} "
                        f"--key {prefix}/payload.bin"
                    ),
                )
            )
    # EC2 instances (critical, hourly).
    for inst_id in (LAB_STATE["instance_a_id"], LAB_STATE["instance_b_id"]):
        if inst_id:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="ec2_instance",
                    id=inst_id,
                    region=REGION,
                    cli_delete_command=f"aws ec2 terminate-instances --instance-ids {inst_id}",
                )
            )
    # NAT Gateway (critical, hourly).
    if LAB_STATE["nat_gateway_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="nat_gateway",
                id=LAB_STATE["nat_gateway_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 delete-nat-gateway --nat-gateway-id {LAB_STATE['nat_gateway_id']}"
                ),
            )
        )
    # EIP (release after NAT delete completes).
    if LAB_STATE["eip_allocation_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="elastic_ip",
                id=LAB_STATE["eip_allocation_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 release-address --allocation-id {LAB_STATE['eip_allocation_id']}"
                ),
            )
        )
    # VPC endpoint.
    if LAB_STATE["vpc_endpoint_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="vpc_endpoint",
                id=LAB_STATE["vpc_endpoint_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 delete-vpc-endpoints --vpc-endpoint-ids {LAB_STATE['vpc_endpoint_id']}"
                ),
            )
        )
    # Route tables (private-a, private-b, public).
    for rt_id in (
        LAB_STATE["private_rt_a_id"],
        LAB_STATE["private_rt_b_id"],
        LAB_STATE["public_rt_id"],
    ):
        if rt_id:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="route_table",
                    id=rt_id,
                    region=REGION,
                    cli_delete_command=f"aws ec2 delete-route-table --route-table-id {rt_id}",
                )
            )
    # Subnets.
    for sn_id in (
        LAB_STATE["private_subnet_a_id"],
        LAB_STATE["private_subnet_b_id"],
        LAB_STATE["public_subnet_id"],
    ):
        if sn_id:
            CLEANUP_MANIFEST.append(
                CleanupResource(
                    type="subnet",
                    id=sn_id,
                    region=REGION,
                    cli_delete_command=f"aws ec2 delete-subnet --subnet-id {sn_id}",
                )
            )
    # Security group.
    if LAB_STATE["sg_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_security_group",
                id=LAB_STATE["sg_id"],
                region=REGION,
                cli_delete_command=f"aws ec2 delete-security-group --group-id {LAB_STATE['sg_id']}",
            )
        )
    # IGW (parent=VPC for detach).
    if LAB_STATE["igw_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="internet_gateway",
                id=LAB_STATE["igw_id"],
                parent=LAB_STATE["vpc_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 detach-internet-gateway --internet-gateway-id "
                    f"{LAB_STATE['igw_id']} --vpc-id {LAB_STATE['vpc_id']} && "
                    f"aws ec2 delete-internet-gateway --internet-gateway-id "
                    f"{LAB_STATE['igw_id']}"
                ),
            )
        )
    # VPC.
    if LAB_STATE["vpc_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="vpc",
                id=LAB_STATE["vpc_id"],
                region=REGION,
                cli_delete_command=f"aws ec2 delete-vpc --vpc-id {LAB_STATE['vpc_id']}",
            )
        )
    # IAM instance profile + role.
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_instance_profile",
            id=INSTANCE_PROFILE_NAME,
            parent=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=(
                f"aws iam remove-role-from-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME} "
                f"--role-name {EC2_ROLE_NAME} && "
                f"aws iam delete-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME}"
            ),
        )
    )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_role",
            id=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=f"aws iam delete-role --role-name {EC2_ROLE_NAME}",
        )
    )
    # S3 bucket.
    if BUCKET_NAME:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="s3_bucket",
                id=BUCKET_NAME,
                region=REGION,
                cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
            )
        )


_rebuild_manifest()


def _atexit_cleanup():
    try:
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision the VPC scaffold.")

## Task 1: Build the shared VPC scaffold

Both architectures live inside one VPC so the comparison is apples to apples. The scaffold has:

- VPC at `10.42.0.0/16` tagged with the lab slug.
- An Internet Gateway attached to the VPC (Architecture A needs it for the NAT Gateway's outbound path).
- A public subnet (`10.42.0.0/24`) in `us-east-1a` for the NAT Gateway.
- A private subnet for Architecture A (`10.42.1.0/24`) in `us-east-1a` so the NAT route works without an AZ hop.
- A private subnet for Architecture B (`10.42.2.0/24`) in `us-east-1b` so you also prove the gateway-endpoint pattern works across AZs.
- A public route table with a default route to the IGW.
- An empty private route table for each side; Tasks 2 and 3 add the routes that define each architecture.

Tag every taggable resource with `labrun:lab-slug=vpc-endpoints-vs-nat-egress` so the orphan scan and cleanup tag audit pick them up.

In [ ]:
# NBVAL_SKIP
# Task 1: create the VPC, IGW, three subnets, and three route tables. Tag
# everything with the lab slug. Capture the IDs in LAB_STATE so the
# manifest rebuild picks them up.

ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def _tag_spec(name, resource_type):
    return {
        "ResourceType": resource_type,
        "Tags": [
            {"Key": "Name", "Value": name},
            {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
        ],
    }


# VPC
# YOUR CODE: call ec2.create_vpc with CidrBlock=VPC_CIDR and
# TagSpecifications=[_tag_spec(VPC_NAME, "vpc")], then capture the VpcId in
# LAB_STATE["vpc_id"]. Wait briefly for the VPC to become available.

# IGW
# YOUR CODE: call ec2.create_internet_gateway with
# TagSpecifications=[_tag_spec(IGW_NAME, "internet-gateway")] and capture the
# InternetGatewayId in LAB_STATE["igw_id"]. Then call
# ec2.attach_internet_gateway(InternetGatewayId=..., VpcId=LAB_STATE["vpc_id"]).

# Public subnet
# YOUR CODE: call ec2.create_subnet with VpcId, CidrBlock=PUBLIC_SUBNET_CIDR,
# AvailabilityZone=AZ_PUBLIC, TagSpecifications=[_tag_spec(PUBLIC_SUBNET_NAME,
# "subnet")]. Capture SubnetId in LAB_STATE["public_subnet_id"].

# Private subnet A
# YOUR CODE: same as public subnet but with PRIVATE_SUBNET_A_CIDR,
# AZ_PRIVATE_A, and PRIVATE_SUBNET_A_NAME. Capture in
# LAB_STATE["private_subnet_a_id"].

# Private subnet B
# YOUR CODE: same as private subnet A but with PRIVATE_SUBNET_B_CIDR,
# AZ_PRIVATE_B, and PRIVATE_SUBNET_B_NAME. Capture in
# LAB_STATE["private_subnet_b_id"].

# Public route table
# YOUR CODE: call ec2.create_route_table with VpcId and
# TagSpecifications=[_tag_spec(PUBLIC_RT_NAME, "route-table")]. Capture the
# RouteTableId in LAB_STATE["public_rt_id"]. Add a default route via the IGW
# with ec2.create_route(RouteTableId=..., DestinationCidrBlock="0.0.0.0/0",
# GatewayId=LAB_STATE["igw_id"]). Then associate it with the public subnet via
# ec2.associate_route_table(SubnetId=LAB_STATE["public_subnet_id"], ...).

# Private route table A (empty for now; Task 2 adds the NAT route).
# YOUR CODE: create_route_table tagged PRIVATE_RT_A_NAME, capture in
# LAB_STATE["private_rt_a_id"], and associate it with the private subnet A.

# Private route table B (empty for now; Task 3 adds the endpoint route).
# YOUR CODE: create_route_table tagged PRIVATE_RT_B_NAME, capture in
# LAB_STATE["private_rt_b_id"], and associate it with the private subnet B.

_rebuild_manifest()
print("VPC scaffold built:")
print(f"  VPC:               {LAB_STATE['vpc_id']}")
print(f"  IGW:               {LAB_STATE['igw_id']}")
print(f"  Public subnet:     {LAB_STATE['public_subnet_id']}")
print(f"  Private subnet A:  {LAB_STATE['private_subnet_a_id']}")
print(f"  Private subnet B:  {LAB_STATE['private_subnet_b_id']}")
print(f"  Public route tbl:  {LAB_STATE['public_rt_id']}")
print(f"  Private route tbl A: {LAB_STATE['private_rt_a_id']}")
print(f"  Private route tbl B: {LAB_STATE['private_rt_b_id']}")

In [ ]:
# NBVAL_SKIP
# Observe: confirm the VPC, IGW attachment, and all three subnets reached
# the available state. Polls every 5 seconds, ceiling 90 seconds.

ec2_obs = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 90
last_summary = None
while time.time() < deadline:
    clear_output(wait=True)
    rows = []
    try:
        vpcs = ec2_obs.describe_vpcs(VpcIds=[LAB_STATE["vpc_id"]]).get("Vpcs", [])
        rows.append(("VPC", LAB_STATE["vpc_id"], vpcs[0].get("State") if vpcs else "?"))
        igws = ec2_obs.describe_internet_gateways(
            InternetGatewayIds=[LAB_STATE["igw_id"]]
        ).get("InternetGateways", [])
        igw_state = "attached" if (igws and igws[0].get("Attachments")) else "detached"
        rows.append(("IGW", LAB_STATE["igw_id"], igw_state))
        subnets = ec2_obs.describe_subnets(
            SubnetIds=[
                LAB_STATE["public_subnet_id"],
                LAB_STATE["private_subnet_a_id"],
                LAB_STATE["private_subnet_b_id"],
            ]
        ).get("Subnets", [])
        for sn in subnets:
            rows.append(("subnet", sn["SubnetId"], sn.get("State")))
    except ClientError as e:
        rows.append(("ERROR", "describe", str(e)))
    print(f"VPC scaffold state at {dt.datetime.now().strftime('%H:%M:%S')}:")
    print("-" * 60)
    for kind, ident, state in rows:
        print(f"  {kind:8}  {ident:30}  {state}")
    last_summary = rows
    states = [r[2] for r in rows if r[0] in ("VPC", "subnet")]
    if all(s == "available" for s in states) and rows[1][2] == "attached":
        print()
        print("All scaffold resources are available. Moving on.")
        break
    time.sleep(5)
else:
    print()
    print("Timed out waiting for the VPC scaffold to settle. Check the AWS console.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: VPC, IGW, public subnet, and both private subnets exist with
# the lab tag and the expected CIDR blocks.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        vpcs = ec2c.describe_vpcs(
            Filters=[{"Name": f"tag:{LAB_TAG_KEY}", "Values": [LAB_TAG_VALUE]}],
        ).get("Vpcs", [])
        if len(vpcs) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected exactly 1 VPC tagged {LAB_TAG_KEY}={LAB_TAG_VALUE}, "
                    f"found {len(vpcs)}."
                ),
            )
        if vpcs[0].get("CidrBlock") != VPC_CIDR:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"VPC CIDR is {vpcs[0].get('CidrBlock')!r}, expected {VPC_CIDR!r}."
                ),
            )

        subnets = ec2c.describe_subnets(
            Filters=[{"Name": f"tag:{LAB_TAG_KEY}", "Values": [LAB_TAG_VALUE]}],
        ).get("Subnets", [])
        cidrs = sorted(s.get("CidrBlock") for s in subnets)
        expected = sorted([PUBLIC_SUBNET_CIDR, PRIVATE_SUBNET_A_CIDR, PRIVATE_SUBNET_B_CIDR])
        if cidrs != expected:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected subnets at CIDRs {expected}, found {cidrs}. "
                    f"All three subnets must carry the lab tag at creation time."
                ),
            )

        igws = ec2c.describe_internet_gateways(
            Filters=[
                {"Name": f"tag:{LAB_TAG_KEY}", "Values": [LAB_TAG_VALUE]},
                {"Name": "attachment.vpc-id", "Values": [vpcs[0]["VpcId"]]},
            ],
        ).get("InternetGateways", [])
        if not igws:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No Internet Gateway is attached to the lab VPC. Architecture "
                    "A needs the IGW for the NAT Gateway's outbound path."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Every taggable VPC resource takes a `TagSpecifications` argument on its `create_*` call. The lab gives you `_tag_spec(name, resource_type)` already; pass it in at creation so the orphan scan and tag audit pick the resource up the moment it exists.

</details>

<details><summary>Hint 2 (direction)</summary>

The order matters. Create the VPC, then the IGW, then call `attach_internet_gateway`. Subnets carry an `AvailabilityZone` argument; use `us-east-1a` for the public subnet and Architecture A's private subnet so the NAT path stays in one AZ, and `us-east-1b` for Architecture B's private subnet. Route tables are created empty (no routes yet) and then associated to subnets via `associate_route_table`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
vpc = ec2.create_vpc(
    CidrBlock=VPC_CIDR,
    TagSpecifications=[_tag_spec(VPC_NAME, "vpc")],
)["Vpc"]
LAB_STATE["vpc_id"] = vpc["VpcId"]
ec2.get_waiter("vpc_available").wait(VpcIds=[LAB_STATE["vpc_id"]])

igw = ec2.create_internet_gateway(
    TagSpecifications=[_tag_spec(IGW_NAME, "internet-gateway")],
)["InternetGateway"]
LAB_STATE["igw_id"] = igw["InternetGatewayId"]
ec2.attach_internet_gateway(
    InternetGatewayId=LAB_STATE["igw_id"], VpcId=LAB_STATE["vpc_id"]
)

def _mk_subnet(name, cidr, az):
    sn = ec2.create_subnet(
        VpcId=LAB_STATE["vpc_id"],
        CidrBlock=cidr,
        AvailabilityZone=az,
        TagSpecifications=[_tag_spec(name, "subnet")],
    )["Subnet"]
    ec2.get_waiter("subnet_available").wait(SubnetIds=[sn["SubnetId"]])
    return sn["SubnetId"]

LAB_STATE["public_subnet_id"] = _mk_subnet(PUBLIC_SUBNET_NAME, PUBLIC_SUBNET_CIDR, AZ_PUBLIC)
LAB_STATE["private_subnet_a_id"] = _mk_subnet(PRIVATE_SUBNET_A_NAME, PRIVATE_SUBNET_A_CIDR, AZ_PRIVATE_A)
LAB_STATE["private_subnet_b_id"] = _mk_subnet(PRIVATE_SUBNET_B_NAME, PRIVATE_SUBNET_B_CIDR, AZ_PRIVATE_B)

def _mk_rt(name, subnet_id):
    rt = ec2.create_route_table(
        VpcId=LAB_STATE["vpc_id"],
        TagSpecifications=[_tag_spec(name, "route-table")],
    )["RouteTable"]
    ec2.associate_route_table(RouteTableId=rt["RouteTableId"], SubnetId=subnet_id)
    return rt["RouteTableId"]

LAB_STATE["public_rt_id"] = _mk_rt(PUBLIC_RT_NAME, LAB_STATE["public_subnet_id"])
ec2.create_route(
    RouteTableId=LAB_STATE["public_rt_id"],
    DestinationCidrBlock="0.0.0.0/0",
    GatewayId=LAB_STATE["igw_id"],
)
LAB_STATE["private_rt_a_id"] = _mk_rt(PRIVATE_RT_A_NAME, LAB_STATE["private_subnet_a_id"])
LAB_STATE["private_rt_b_id"] = _mk_rt(PRIVATE_RT_B_NAME, LAB_STATE["private_subnet_b_id"])
```

</details>

**Wallet check.** Still at $0.00. VPCs, subnets, route tables, IGWs, and security groups are all free. The bill does not move until you add the NAT Gateway in Task 2.

## Task 2: Build Architecture A (NAT Gateway path)

Architecture A is the wrong-but-common pattern. Build it like the previous lead did, but tagged for cleanup so the bill stops when you do.

Three calls:

1. `allocate_address(Domain="vpc")` to get an Elastic IP for the NAT Gateway. Tag it with the lab slug.
2. `create_nat_gateway(SubnetId=public_subnet_id, AllocationId=...)` so the gateway lives in the public subnet with the EIP attached. Tag it with the lab slug.
3. After the NAT Gateway reaches `State=available` (~2-3 minutes), add a `0.0.0.0/0` route on `private-rt-a` pointing at the NAT Gateway.

The observe cell below polls until the NAT Gateway is available, prints the state on each tick, and breaks out the moment it transitions. Plan for two to three minutes.

In [ ]:
# NBVAL_SKIP
# Task 2: Architecture A. Allocate the EIP, create the NAT Gateway in the
# public subnet, and route private-rt-a through it.

# Allocate the EIP.
# YOUR CODE: call ec2.allocate_address(Domain="vpc", TagSpecifications=[_tag_spec(
# f"labrun-{LAB_ID}-eip", "elastic-ip")]). Capture AllocationId in
# LAB_STATE["eip_allocation_id"].

# Create the NAT Gateway in the public subnet with the EIP attached.
# YOUR CODE: call ec2.create_nat_gateway(SubnetId=LAB_STATE["public_subnet_id"],
# AllocationId=LAB_STATE["eip_allocation_id"], TagSpecifications=[
# _tag_spec(f"labrun-{LAB_ID}-nat-gw", "natgateway")]). Capture NatGatewayId in
# LAB_STATE["nat_gateway_id"]. The route insertion happens after the observe
# cell confirms State=available.

_rebuild_manifest()
print(f"EIP allocated: {LAB_STATE['eip_allocation_id']}")
print(f"NAT Gateway requested: {LAB_STATE['nat_gateway_id']}")
print("Now run the observe cell below to watch the NAT Gateway transition")
print("to State=available (typically 2-3 minutes).")

In [ ]:
# NBVAL_SKIP
# Observe: poll the NAT Gateway until State=available. After it lands, insert
# the 0.0.0.0/0 route on private-rt-a so Architecture A is complete.

ec2_obs = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 300  # 5-minute ceiling
nat_state = "pending"
while time.time() < deadline:
    clear_output(wait=True)
    try:
        gw = ec2_obs.describe_nat_gateways(
            NatGatewayIds=[LAB_STATE["nat_gateway_id"]]
        )["NatGateways"][0]
        nat_state = gw.get("State")
        addresses = gw.get("NatGatewayAddresses", [{}])
        public_ip = addresses[0].get("PublicIp", "?")
    except (ClientError, IndexError) as e:
        nat_state = f"error: {e}"
        public_ip = "?"
    elapsed = int(300 - (deadline - time.time()))
    print(f"NAT Gateway status at t+{elapsed:>3}s:")
    print("-" * 60)
    print(f"  ID:        {LAB_STATE['nat_gateway_id']}")
    print(f"  State:     {nat_state}")
    print(f"  EIP:       {public_ip}")
    print(f"  Allocation:{LAB_STATE['eip_allocation_id']}")
    if nat_state == "available":
        print()
        print("NAT Gateway is available. Inserting 0.0.0.0/0 route on private-rt-a...")
        try:
            ec2_obs.create_route(
                RouteTableId=LAB_STATE["private_rt_a_id"],
                DestinationCidrBlock="0.0.0.0/0",
                NatGatewayId=LAB_STATE["nat_gateway_id"],
            )
            print("Route created. Architecture A is complete.")
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            if code == "RouteAlreadyExists":
                print("Route already present; skipping.")
            else:
                print(f"Route creation failed: {e}")
        break
    if nat_state in ("failed", "deleting", "deleted"):
        print()
        print(f"NAT Gateway entered terminal {nat_state!r} state. Check the AWS console.")
        break
    time.sleep(15)
else:
    print()
    print("Timed out waiting for NAT Gateway to reach available state.")
    print("Run `aws ec2 describe-nat-gateways` to investigate.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: NAT Gateway is available with the EIP attached and
# private-rt-a has the 0.0.0.0/0-to-NAT route.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        gws = ec2c.describe_nat_gateways(
            Filter=[{"Name": "tag:" + LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
        ).get("NatGateways", [])
        live = [g for g in gws if g.get("State") == "available"]
        if not live:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No NAT Gateway in State=available tagged for this lab. "
                    "It may still be provisioning (typical 2-3 minutes)."
                ),
            )
        gw = live[0]
        if gw.get("SubnetId") != LAB_STATE["public_subnet_id"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NAT Gateway is in subnet {gw.get('SubnetId')}, not the "
                    f"public subnet {LAB_STATE['public_subnet_id']}."
                ),
            )
        addresses = gw.get("NatGatewayAddresses", [])
        if not addresses or addresses[0].get("AllocationId") != LAB_STATE["eip_allocation_id"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "NAT Gateway is not using the lab EIP allocation. "
                    f"Expected {LAB_STATE['eip_allocation_id']}, "
                    f"got {addresses[0].get('AllocationId') if addresses else None}."
                ),
            )

        rt = ec2c.describe_route_tables(
            RouteTableIds=[LAB_STATE["private_rt_a_id"]]
        )["RouteTables"][0]
        nat_route = next(
            (
                r
                for r in rt.get("Routes", [])
                if r.get("DestinationCidrBlock") == "0.0.0.0/0"
                and r.get("NatGatewayId") == LAB_STATE["nat_gateway_id"]
            ),
            None,
        )
        if nat_route is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "private-rt-a does not carry a 0.0.0.0/0 route pointing at "
                    "the lab NAT Gateway. Add it with ec2.create_route."
                ),
            )
        assoc = next(
            (a for a in rt.get("Associations", [])
             if a.get("SubnetId") == LAB_STATE["private_subnet_a_id"]),
            None,
        )
        if assoc is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "private-rt-a is not associated with private-subnet-a. "
                    "Run ec2.associate_route_table to fix it."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

NAT Gateway creation is asynchronous. Three things have to land before Checkpoint 2 will pass: the gateway reaches `State=available`, the EIP is referenced via `AllocationId` on the gateway, and `private-rt-a` carries a `0.0.0.0/0` route pointing at the gateway.

</details>

<details><summary>Hint 2 (direction)</summary>

The route call is the part that is easy to skip. After the observe cell prints `State=available`, the lab inserts the route automatically. If you skipped the observe cell, run `ec2.create_route(RouteTableId=LAB_STATE["private_rt_a_id"], DestinationCidrBlock="0.0.0.0/0", NatGatewayId=LAB_STATE["nat_gateway_id"])` yourself.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
eip = ec2.allocate_address(
    Domain="vpc",
    TagSpecifications=[_tag_spec(f"labrun-{LAB_ID}-eip", "elastic-ip")],
)
LAB_STATE["eip_allocation_id"] = eip["AllocationId"]

nat = ec2.create_nat_gateway(
    SubnetId=LAB_STATE["public_subnet_id"],
    AllocationId=LAB_STATE["eip_allocation_id"],
    TagSpecifications=[_tag_spec(f"labrun-{LAB_ID}-nat-gw", "natgateway")],
)
LAB_STATE["nat_gateway_id"] = nat["NatGateway"]["NatGatewayId"]
```

The observe cell waits for `State=available` and then inserts the `0.0.0.0/0` route on `private-rt-a` for you.

</details>

**Wallet check.** NAT Gateway just started billing. $0.045/hour means about 0.075 cents per minute. A 60-minute lab from this point forward costs about 4.5 cents in NAT hours, plus the $0.045/GB the gateway charges for the 100 MB Architecture A is about to push through it (~half a cent). The cleanup cell at the bottom is what stops the meter; do not close the tab without running it.

## Task 3: Build Architecture B (VPC Gateway Endpoint for S3)

This is the right pattern. Two calls do the work the NAT Gateway path took five resources to express.

1. Look up the S3 prefix list ID via `describe_prefix_lists(Filters=[Name=prefix-list-name, Values=com.amazonaws.us-east-1.s3])`. The prefix list is a managed list of S3 IP ranges in this region; the route table will point at it instead of `0.0.0.0/0`.
2. `create_vpc_endpoint(ServiceName="com.amazonaws.us-east-1.s3", VpcId=..., VpcEndpointType="Gateway", RouteTableIds=[private_rt_b_id])`. The `RouteTableIds` argument injects the prefix-list route into `private-rt-b` automatically.

Confirm `private-rt-b` does NOT also carry a `0.0.0.0/0` route. The whole point of Architecture B is no internet egress at all; the design intent is S3-only.

In [ ]:
# NBVAL_SKIP
# Task 3: Architecture B. Look up the S3 prefix list, create the gateway
# endpoint pointed at private-rt-b, and tag it for cleanup.

# Look up the S3 prefix list ID for this region.
# YOUR CODE: call ec2.describe_prefix_lists(Filters=[{"Name": "prefix-list-name",
# "Values": [f"com.amazonaws.{REGION}.s3"]}]) and capture the PrefixListId in a
# local variable s3_prefix_list_id. Print it so you can see what AWS returned.

s3_prefix_list_id = None  # YOUR CODE replaces this with the lookup result.

# Create the gateway endpoint with private-rt-b attached.
# YOUR CODE: call ec2.create_vpc_endpoint(ServiceName=f"com.amazonaws.{REGION}.s3",
# VpcId=LAB_STATE["vpc_id"], VpcEndpointType="Gateway",
# RouteTableIds=[LAB_STATE["private_rt_b_id"]], TagSpecifications=[_tag_spec(
# f"labrun-{LAB_ID}-vpc-endpoint-s3", "vpc-endpoint")]). Capture VpcEndpointId
# in LAB_STATE["vpc_endpoint_id"].

_rebuild_manifest()
print(f"S3 prefix list ID: {s3_prefix_list_id}")
print(f"VPC endpoint:      {LAB_STATE['vpc_endpoint_id']}")
print()
print("Run the observe cell below to confirm the endpoint reaches")
print("State=available and the route lands on private-rt-b.")

In [ ]:
# NBVAL_SKIP
# Observe: poll the gateway endpoint until State=available. Typically 10-30
# seconds; ceiling 90 seconds.

ec2_obs = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 90
endpoint_state = "pending"
while time.time() < deadline:
    clear_output(wait=True)
    try:
        ep = ec2_obs.describe_vpc_endpoints(
            VpcEndpointIds=[LAB_STATE["vpc_endpoint_id"]]
        )["VpcEndpoints"][0]
        endpoint_state = ep.get("State")
        route_tables = ep.get("RouteTableIds", [])
    except (ClientError, IndexError) as e:
        endpoint_state = f"error: {e}"
        route_tables = []
    elapsed = int(90 - (deadline - time.time()))
    print(f"VPC Gateway Endpoint at t+{elapsed:>3}s:")
    print("-" * 60)
    print(f"  ID:           {LAB_STATE['vpc_endpoint_id']}")
    print(f"  State:        {endpoint_state}")
    print(f"  RouteTables:  {route_tables}")
    if endpoint_state == "available" and LAB_STATE["private_rt_b_id"] in route_tables:
        print()
        print("Gateway endpoint is available and routed through private-rt-b.")
        break
    if endpoint_state in ("Failed", "Deleting", "Deleted"):
        print()
        print(f"Endpoint entered terminal {endpoint_state!r} state. Check the AWS console.")
        break
    time.sleep(5)
else:
    print()
    print("Timed out waiting for the gateway endpoint to reach available.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Gateway endpoint exists, private-rt-b carries the S3 prefix
# list route, and private-rt-b has NO 0.0.0.0/0 route.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        eps = ec2c.describe_vpc_endpoints(
            Filters=[{"Name": "tag:" + LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
        ).get("VpcEndpoints", [])
        ep = next(
            (e for e in eps if e.get("VpcEndpointType") == "Gateway"
             and e.get("ServiceName") == f"com.amazonaws.{REGION}.s3"),
            None,
        )
        if ep is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No Gateway-type S3 VPC endpoint tagged for this lab. "
                    f"ServiceName must be com.amazonaws.{REGION}.s3."
                ),
            )
        if ep.get("State") != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"VPC endpoint state is {ep.get('State')!r}, expected available."
                ),
            )
        if LAB_STATE["private_rt_b_id"] not in ep.get("RouteTableIds", []):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "VPC endpoint is not attached to private-rt-b. Re-create the "
                    "endpoint with RouteTableIds=[private-rt-b]."
                ),
            )

        rt = ec2c.describe_route_tables(
            RouteTableIds=[LAB_STATE["private_rt_b_id"]]
        )["RouteTables"][0]
        prefix_route = next(
            (
                r
                for r in rt.get("Routes", [])
                if r.get("GatewayId") == ep["VpcEndpointId"]
            ),
            None,
        )
        if prefix_route is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "private-rt-b is missing the S3 prefix-list route pointing at "
                    "the gateway endpoint."
                ),
            )

        bad_route = next(
            (
                r
                for r in rt.get("Routes", [])
                if r.get("DestinationCidrBlock") == "0.0.0.0/0"
            ),
            None,
        )
        if bad_route is not None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "private-rt-b carries a 0.0.0.0/0 route. Architecture B's "
                    "design intent is S3-only, no internet egress. Delete that "
                    "route with ec2.delete_route."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Two boto3 calls. The first looks up the prefix list ID for `com.amazonaws.us-east-1.s3`. The second creates a gateway endpoint that knows about your VPC, the S3 service name, and the private route table it should route through.

</details>

<details><summary>Hint 2 (direction)</summary>

`describe_prefix_lists(Filters=[{"Name": "prefix-list-name", "Values": ["com.amazonaws.us-east-1.s3"]}])` returns a `PrefixLists` array; pull `PrefixListId` from the first entry. `create_vpc_endpoint(VpcEndpointType="Gateway", RouteTableIds=[...])` is the call that wires the prefix-list route automatically; you do not call `create_route` here.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
plr = ec2.describe_prefix_lists(
    Filters=[{"Name": "prefix-list-name", "Values": [f"com.amazonaws.{REGION}.s3"]}],
)["PrefixLists"]
s3_prefix_list_id = plr[0]["PrefixListId"]

ep = ec2.create_vpc_endpoint(
    ServiceName=f"com.amazonaws.{REGION}.s3",
    VpcId=LAB_STATE["vpc_id"],
    VpcEndpointType="Gateway",
    RouteTableIds=[LAB_STATE["private_rt_b_id"]],
    TagSpecifications=[_tag_spec(f"labrun-{LAB_ID}-vpc-endpoint-s3", "vpc-endpoint")],
)
LAB_STATE["vpc_endpoint_id"] = ep["VpcEndpoint"]["VpcEndpointId"]
```

</details>

**Wallet check.** VPC Gateway Endpoints are free. The bill is still moving from the NAT Gateway in Architecture A, not from anything you just created. About six cents so far on a clean run.

## Task 4: Launch one EC2 instance in each private subnet and upload 100 MB

Each instance pulls the same Amazon Linux 2023 ARM AMI, runs the same user-data, and ends up with the same 100 MB payload in S3. The only difference is which path the upload took.

You need:

- A shared IAM role + instance profile granting `s3:PutObject` on the lab bucket.
- A shared security group (no inbound, all outbound) that both instances use.
- Two t4g.nano launches with the user-data below.

The user-data does three things: generate 100 MB of zeros with `dd`, save it to `/tmp/p.bin`, and upload it via the AWS CLI to `s3://{BUCKET_NAME}/from-instance-{a or b}/payload.bin`. The CLI is pre-installed on AL2023, so user-data does not need to bootstrap it.

The observe cell polls until both instances reach `Status=ok` (user-data has had time to execute) and both S3 objects appear. Plan for 4-6 minutes; the EC2 status check itself takes 2-3 minutes after `running`.

In [ ]:
# NBVAL_SKIP
# Task 4: IAM, security group, and launch both instances.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ssm = boto3.client(
    "ssm",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the S3 bucket. The cleanup s3_bucket adapter empties + deletes it.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(
        Bucket=BUCKET_NAME,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") not in ("BucketAlreadyOwnedByYou",):
        raise

# IAM role with EC2 assume-role trust and an inline s3:PutObject policy.
trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "ec2.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=EC2_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
        raise

inline = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:PutObject"],
            "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*",
        }
    ],
}
iam.put_role_policy(
    RoleName=EC2_ROLE_NAME,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline),
)

try:
    iam.create_instance_profile(
        InstanceProfileName=INSTANCE_PROFILE_NAME,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
        raise

try:
    iam.add_role_to_instance_profile(
        InstanceProfileName=INSTANCE_PROFILE_NAME, RoleName=EC2_ROLE_NAME
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "LimitExceeded":
        raise

# Security group: no inbound, all outbound. The default outbound rule covers
# both HTTPS to S3 and the NAT path.
try:
    sg = ec2.create_security_group(
        GroupName=SG_NAME,
        Description="labrun shared SG for VPC endpoints vs NAT egress lab",
        VpcId=LAB_STATE["vpc_id"],
        TagSpecifications=[_tag_spec(SG_NAME, "security-group")],
    )
    LAB_STATE["sg_id"] = sg["GroupId"]
    # Drop the default 0.0.0.0/0 ingress so the SG is properly tight.
    # (boto3 default-VPC SG creation does not add ingress; only the egress
    # default exists, which we keep.)
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_NAME]}],
        )["SecurityGroups"]
        LAB_STATE["sg_id"] = sgs[0]["GroupId"]
    else:
        raise

# Resolve the latest AL2023 ARM AMI via SSM.
ami_id = ssm.get_parameter(
    Name="/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-arm64",
)["Parameter"]["Value"]
print(f"AL2023 ARM AMI: {ami_id}")

# IAM eventual consistency: wait ~15 seconds before the run_instances call so
# the instance profile is visible to EC2.
print("Waiting 15s for IAM instance profile to propagate...")
time.sleep(15)


def _user_data(prefix):
    script = (
        "#!/bin/bash\n"
        "set -e\n"
        f"dd if=/dev/zero of=/tmp/p.bin bs=1M count=100\n"
        f"aws s3 cp /tmp/p.bin s3://{BUCKET_NAME}/{prefix}/payload.bin "
        f"--region {REGION}\n"
    )
    return base64.b64encode(script.encode("utf-8")).decode("utf-8")


def _launch(name, subnet_id, prefix):
    # YOUR CODE: call ec2.run_instances with ImageId=ami_id, InstanceType="t4g.nano",
    # MinCount=1, MaxCount=1, SubnetId=subnet_id, SecurityGroupIds=[LAB_STATE["sg_id"]],
    # IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME},
    # UserData=_user_data(prefix), and TagSpecifications=[_tag_spec(name, "instance")].
    # Return the launched InstanceId.
    pass


LAB_STATE["instance_a_id"] = _launch(EC2_A_NAME, LAB_STATE["private_subnet_a_id"], "from-instance-a")
LAB_STATE["instance_b_id"] = _launch(EC2_B_NAME, LAB_STATE["private_subnet_b_id"], "from-instance-b")
_rebuild_manifest()

print(f"Instance A: {LAB_STATE['instance_a_id']}")
print(f"Instance B: {LAB_STATE['instance_b_id']}")
print()
print("Run the observe cell below to watch both instances reach Status=ok and")
print("both S3 objects appear (typically 4-6 minutes total).")

In [ ]:
# NBVAL_SKIP
# Observe: poll until both instances reach Status=ok and both 100 MB payloads
# show up in S3. Ceiling 12 minutes (instances take 2-3 min to reach running,
# another 2-3 min for status checks, then dd + 100 MB upload).

ec2_obs = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
s3_obs = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 720
targets = [
    ("Instance A", LAB_STATE["instance_a_id"], "from-instance-a/payload.bin"),
    ("Instance B", LAB_STATE["instance_b_id"], "from-instance-b/payload.bin"),
]
while time.time() < deadline:
    clear_output(wait=True)
    elapsed = int(720 - (deadline - time.time()))
    print(f"Architecture A and B launch progress at t+{elapsed:>3}s:")
    print("-" * 78)
    print(f"  {'side':10}  {'instance':22}  {'state':14}  {'status':10}  {'s3 size':10}")
    print("-" * 78)
    done_count = 0
    for side, inst_id, key in targets:
        try:
            r = ec2_obs.describe_instances(InstanceIds=[inst_id])["Reservations"][0]["Instances"][0]
            state = r["State"]["Name"]
        except Exception as e:
            state = f"err: {e}"
        try:
            status = ec2_obs.describe_instance_status(
                InstanceIds=[inst_id], IncludeAllInstances=True
            )["InstanceStatuses"][0]["InstanceStatus"]["Status"]
        except Exception:
            status = "n/a"
        try:
            head = s3_obs.head_object(Bucket=BUCKET_NAME, Key=key)
            size = head["ContentLength"]
            size_str = f"{size/1_000_000:.1f} MB"
            if size > 95_000_000:
                done_count += 1
        except ClientError:
            size_str = "missing"
        print(f"  {side:10}  {inst_id:22}  {state:14}  {status:10}  {size_str:10}")
    if done_count == 2:
        print()
        print("Both 100 MB payloads are in S3. Architecture A used the NAT path;")
        print("Architecture B used the gateway endpoint.")
        break
    time.sleep(15)
else:
    print()
    print("Timed out before both payloads landed. Check the EC2 console for")
    print("user-data logs (Actions > Monitor and troubleshoot > Get system log).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Both EC2 instances are running with the right subnets and
# both 100 MB payload objects exist in S3.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        for side, inst_id, expected_subnet in (
            ("A", LAB_STATE["instance_a_id"], LAB_STATE["private_subnet_a_id"]),
            ("B", LAB_STATE["instance_b_id"], LAB_STATE["private_subnet_b_id"]),
        ):
            if not inst_id:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Instance {side} was not launched.",
                )
            r = ec2c.describe_instances(InstanceIds=[inst_id])["Reservations"]
            if not r or not r[0]["Instances"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Instance {side} ({inst_id}) is no longer described by EC2.",
                )
            inst = r[0]["Instances"][0]
            if inst["State"]["Name"] != "running":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Instance {side} ({inst_id}) is in state "
                        f"{inst['State']['Name']!r}, expected running."
                    ),
                )
            if inst.get("SubnetId") != expected_subnet:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Instance {side} is in subnet {inst.get('SubnetId')}, "
                        f"expected {expected_subnet}."
                    ),
                )

        for side, key in (
            ("A", "from-instance-a/payload.bin"),
            ("B", "from-instance-b/payload.bin"),
        ):
            try:
                head = s3c.head_object(Bucket=BUCKET_NAME, Key=key)
            except ClientError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Instance {side} payload is missing at s3://{BUCKET_NAME}/{key}. "
                        f"User-data may still be running or the IAM PutObject grant "
                        f"is wrong: {e}"
                    ),
                )
            size = head.get("ContentLength", 0)
            if size < 95_000_000 or size > 110_000_000:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Instance {side} payload size is {size} bytes, expected "
                        f"~100 MB. The dd in user-data may have been truncated."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The `_launch` helper is missing its `run_instances` body. Every argument it needs is already in the surrounding code (subnet, SG, instance profile, AMI, user-data). Wire them up and return the new instance ID.

</details>

<details><summary>Hint 2 (direction)</summary>

`ec2.run_instances` returns `{"Instances": [...]}`. The `InstanceId` lives on the first element. Pass `IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME}` (not `{"Arn": ...}`) so EC2 looks the profile up by name and you do not need to compute the ARN. The status check (`Status=ok`) is what guarantees user-data had time to run; the observe cell waits for that signal.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
def _launch(name, subnet_id, prefix):
    resp = ec2.run_instances(
        ImageId=ami_id,
        InstanceType="t4g.nano",
        MinCount=1,
        MaxCount=1,
        SubnetId=subnet_id,
        SecurityGroupIds=[LAB_STATE["sg_id"]],
        IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME},
        UserData=_user_data(prefix),
        TagSpecifications=[_tag_spec(name, "instance")],
    )
    return resp["Instances"][0]["InstanceId"]
```

</details>

**Wallet check.** Two t4g.nano instances at $0.0042/hour each (less than half a cent per hour combined). The NAT Gateway just processed 100 MB at $0.045/GB, costing about half a cent. Running total: roughly 7 to 8 cents.

## Task 5: Read the CloudWatch NAT Gateway BytesOutToDestination metric

This is the punchline. The NAT Gateway has been billing for processed bytes, and CloudWatch has the receipts.

The metric you want is `AWS/NATGateway BytesOutToDestination` with `NatGatewayId` as the only dimension. Pull it with a 5-minute period and a `Sum` statistic. The window starts at `LAB_STATE["session_start"]` (captured in the setup cell) and ends now.

The number should land somewhere between 80 MB and 150 MB:

- 80 MB lower bound: the 100 MB dd minus AWS's metric publishing granularity.
- 150 MB upper bound: 100 MB plus retries, protocol overhead, AWS CLI multipart upload control traffic.

Architecture B's traffic never touched the NAT Gateway. Its bytes are zero by construction; you do not read a separate metric for it. The route table on `private-rt-b` is the proof.

CloudWatch publishes NAT Gateway metrics at 1-minute granularity but the 5-minute Sum window needs to close before the aggregate stabilizes. The observe cell waits up to 6 minutes after the upload completes.

In [ ]:
# NBVAL_SKIP
# Task 5: Read NAT Gateway BytesOutToDestination over the session window.

cloudwatch = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def _read_nat_bytes():
    # YOUR CODE: call cloudwatch.get_metric_statistics with
    # Namespace="AWS/NATGateway", MetricName="BytesOutToDestination",
    # Dimensions=[{"Name": "NatGatewayId", "Value": LAB_STATE["nat_gateway_id"]}],
    # StartTime=LAB_STATE["session_start"], EndTime=dt.datetime.now(dt.timezone.utc),
    # Period=300, Statistics=["Sum"]. Sum the "Sum" field across all returned
    # datapoints and return total bytes as an int.
    return 0


total_bytes = _read_nat_bytes()
print(f"NAT Gateway processed: {total_bytes:,} bytes ({total_bytes/1_000_000:.1f} MB)")
print(f"VPC Endpoint path:     0 bytes through NAT (route-table-proven)")

In [ ]:
# NBVAL_SKIP
# Observe: poll the CloudWatch metric until the Sum lands in the 80-150 MB
# range. CloudWatch NAT Gateway metrics publish at 1-minute granularity but
# aggregation needs the 5-minute window to close. Ceiling 6 minutes.

cw_obs = boto3.client(
    "cloudwatch",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 360
total_bytes = 0
while time.time() < deadline:
    clear_output(wait=True)
    try:
        resp = cw_obs.get_metric_statistics(
            Namespace="AWS/NATGateway",
            MetricName="BytesOutToDestination",
            Dimensions=[{"Name": "NatGatewayId", "Value": LAB_STATE["nat_gateway_id"]}],
            StartTime=LAB_STATE["session_start"],
            EndTime=dt.datetime.now(dt.timezone.utc),
            Period=300,
            Statistics=["Sum"],
        )
        total_bytes = int(sum(dp["Sum"] for dp in resp.get("Datapoints", [])))
    except ClientError as e:
        total_bytes = -1
    elapsed = int(360 - (deadline - time.time()))
    print(f"NAT Gateway BytesOutToDestination Sum at t+{elapsed:>3}s:")
    print("-" * 60)
    print(f"  NAT Gateway:        {LAB_STATE['nat_gateway_id']}")
    print(f"  Bytes through NAT:  {total_bytes:,} bytes ({total_bytes/1_000_000:.1f} MB)")
    print(f"  VPC Endpoint path:  0 bytes through NAT (route-table-proven)")
    if 80_000_000 <= total_bytes <= 150_000_000:
        print()
        print("Metric is in the expected 80-150 MB range. Architecture A pushed")
        print("the upload through NAT; Architecture B's bytes never appear here.")
        break
    time.sleep(30)
else:
    print()
    print("Timed out. The metric may not have published yet; try the checkpoint")
    print("again in another 60-90 seconds if it does not pass.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: NAT Gateway BytesOutToDestination Sum across the session
# window is between 80 MB and 150 MB. Per blueprint Section 21, the
# comparison logic lives in the reflection MCQ; this checkpoint is an
# objective metric capture.

def checkpoint_5(session):
    from labrun_checks import CheckpointResult
    try:
        cw = boto3.client(
            "cloudwatch",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        resp = cw.get_metric_statistics(
            Namespace="AWS/NATGateway",
            MetricName="BytesOutToDestination",
            Dimensions=[{"Name": "NatGatewayId", "Value": LAB_STATE["nat_gateway_id"]}],
            StartTime=LAB_STATE["session_start"],
            EndTime=dt.datetime.now(dt.timezone.utc),
            Period=300,
            Statistics=["Sum"],
        )
        total = int(sum(dp["Sum"] for dp in resp.get("Datapoints", [])))
        if total < 80_000_000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NAT BytesOutToDestination Sum is {total:,} bytes "
                    f"({total/1_000_000:.1f} MB), below the 80 MB floor. "
                    f"CloudWatch may still be aggregating; wait 60-90 seconds "
                    f"and re-run, or confirm the Architecture A upload finished."
                ),
            )
        if total > 150_000_000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NAT BytesOutToDestination Sum is {total:,} bytes, above "
                    f"the 150 MB ceiling. Did the user-data upload run more than "
                    f"once? Each instance should upload exactly one 100 MB object."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

`get_metric_statistics` needs five things: the namespace, the metric name, the dimension, the time window, and the period plus statistic. All five are already in scope (constants, `LAB_STATE`, `dt.datetime`).

</details>

<details><summary>Hint 2 (direction)</summary>

`Period=300` is a 5-minute aggregation bucket. With `Statistics=["Sum"]`, each bucket gives you the total bytes that left the NAT during that 5 minutes. Sum all buckets to get the total for the session. Some sessions span a single bucket, some span two; the sum handles both.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
def _read_nat_bytes():
    resp = cloudwatch.get_metric_statistics(
        Namespace="AWS/NATGateway",
        MetricName="BytesOutToDestination",
        Dimensions=[{"Name": "NatGatewayId", "Value": LAB_STATE["nat_gateway_id"]}],
        StartTime=LAB_STATE["session_start"],
        EndTime=dt.datetime.now(dt.timezone.utc),
        Period=300,
        Statistics=["Sum"],
    )
    return int(sum(dp["Sum"] for dp in resp.get("Datapoints", [])))
```

</details>

**Wallet check.** No new line items. CloudWatch `GetMetricStatistics` calls are billed at $0.01 per 1,000 requests after the free tier; you spent a handful, well inside the free million. Running total still about 8 to 10 cents.

## Cleanup

This is the one that matters. The NAT Gateway is still billing at 4.5 cents per hour until the cleanup cell runs. Critical-first ordering means EC2 instances and the NAT Gateway tear down before the network scaffold. Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Critical-first reverse-creation order. The NAT Gateway delete
# waits for State=deleted before the EIP release fires so the unattached-EIP
# billing window stays at zero.

import sys

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(
    1
    for r in CLEANUP_MANIFEST
    if r.type in ("ec2_instance", "nat_gateway")
)
critical_destroyed = sum(
    1
    for r in CLEANUP_MANIFEST
    if r.type in ("ec2_instance", "nat_gateway") and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1
    for rid in failed_ids
    for r in CLEANUP_MANIFEST
    if r.id == rid and r.type not in ("ec2_instance", "nat_gateway")
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.05 to $0.15.** NAT Gateway hours plus ~100 MB of NAT-processed bytes account for the bulk; two t4g.nano hours add about a cent; S3 PUT requests round to zero. The cleanup cell above just terminated everything, so the meter stopped the moment that cell printed `Cleanup complete`. The EIP was released within seconds of the NAT Gateway reaching deleted state, so the unattached-EIP $0.005/hour rate never kicked in.

## Reflection

These are not graded. They are for you.

1. Walk through every cost line item on the NAT Gateway path for a workload that uploads 1 TB of compressed logs to S3 per day. List the hourly NAT fee, the per-GB data-processing fee, and any data-transfer-out fees. What is the monthly bill, and how much of it disappears if you replace the NAT Gateway with an S3 gateway endpoint?

2. You have a private workload that needs to call S3, DynamoDB, Kinesis Data Streams, and Secrets Manager. Which of those services support gateway endpoints (free) and which only support interface endpoints (paid)? For the interface-endpoint services, when is an interface endpoint cheaper than a NAT Gateway, and when is it not?

## Exam-Style Practice

**Question 1 (Domain 4, NAT vs VPC endpoint cost optimization):**

A team runs an analytics workload in a private subnet that uploads 200 GB of compressed logs to S3 per day. The current architecture routes that traffic through a NAT Gateway. The team wants to keep the workload private (no public IPs, no internet egress) and minimize cost. Which change has the largest cost impact?

A. Replace the NAT Gateway with a VPC Gateway Endpoint for S3 and remove the 0.0.0.0/0 route from the private subnet's route table.

B. Move the NAT Gateway to a Spot-pricing-equivalent instance type and accept occasional interruptions.

C. Add an Application Load Balancer in front of S3 to cache repeated uploads at the edge.

D. Switch to a NAT Instance on a t4g.nano to save on the NAT Gateway hourly fee while keeping the data-processing path identical.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: VPC Gateway Endpoints for S3 are free (no hourly fee, no per-GB data-processing fee). Replacing the NAT path eliminates the $0.045/hour NAT fee plus the $0.045/GB data-processing fee on every byte. For 200 GB/day that is approximately $32/day of NAT data-processing alone, gone.
- B is wrong: NAT Gateways do not have a Spot pricing option. NAT Gateway is a managed service with a single on-demand price.
- C is wrong: ALBs do not front S3 in a way that caches uploads. ALBs are HTTP/HTTPS load balancers for EC2/IP/Lambda targets. CloudFront could front S3 for downloads, but the workload here is uploads, not downloads.
- D is partially right that a NAT Instance is cheaper per hour than a NAT Gateway, but NAT Instances are self-managed, scale poorly, and are the AWS-deprecated pattern. The right pick is to eliminate the NAT path entirely for S3 traffic via the gateway endpoint.

</details>

**Question 2 (Domain 1, perimeter security with VPC endpoints):**

A regulated workload runs in a private subnet and writes encrypted reports to S3. The compliance team requires:

- No public internet egress from the workload's subnet (no NAT path, no IGW path).
- Audit-able proof in CloudTrail that S3 traffic stayed inside the AWS network and never traversed the public internet.
- No per-hour fee for the egress path.

Which architecture satisfies all three requirements?

A. NAT Gateway in the public subnet plus a default route from the private subnet, with S3 access via the public S3 endpoint.

B. VPC Gateway Endpoint for S3 attached to the private subnet's route table, with no 0.0.0.0/0 route present.

C. VPC Interface Endpoint (PrivateLink) for S3 in the private subnet, with no other egress routes.

D. Direct Connect from the data center to AWS, with S3 traffic routed over the Direct Connect link.

<details><summary>Show answer</summary>

**Correct: B.**

- A fails the "no public internet egress" requirement: a NAT Gateway IS public egress through the AWS-owned IP space. The traffic to S3 traverses the internet-facing S3 endpoint.
- B is correct: a VPC Gateway Endpoint keeps S3 traffic on the AWS internal network (it never traverses the public internet), it adds CloudTrail VPC-endpoint events that the compliance team can audit, it is free of per-hour charges, and it does not require a 0.0.0.0/0 route. Three for three.
- C is wrong on the "no per-hour fee" requirement: VPC Interface Endpoints (PrivateLink) bill $0.01/hour per endpoint per AZ. Gateway endpoints (the right answer for S3 and DynamoDB) are free; interface endpoints are paid.
- D is wrong: Direct Connect is a network link from a customer location to AWS, not an in-VPC egress path. It does not apply to a workload running inside a private subnet talking to S3.

</details>